[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EddiazOS/LC_data_review/blob/main/05_Analisis_Reproducible/notebooks/05_nivel_5_evaluacion_riesgo_salud.ipynb)

# Notebook 05: Nivel 5 — Evaluación Probabilística de Riesgos en Salud
### Subsección 3.5 del Manuscrito: Simulación Monte Carlo (10,000 iteraciones), EDIs, HQ e ILCR
**Proyecto LC-GC / Indoor Dust — Universidad de Cartagena & Universidad de Amberes**


In [ ]:
# Celda 1: Configuración de Entorno (Google Colab / Local) y Pre-requisitos
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🚀 Entorno Google Colab detectado.")
    if not os.path.exists('LC_data_review'):
        !git clone https://github.com/EddiazOS/LC_data_review.git
    if os.path.exists('LC_data_review/05_Analisis_Reproducible/datos_curados'):
        DATA_DIR = 'LC_data_review/05_Analisis_Reproducible/datos_curados'
        OUT_TABLES = 'LC_data_review/05_Analisis_Reproducible/outputs/tablas'
        OUT_FIGS = 'LC_data_review/05_Analisis_Reproducible/outputs/figuras'
    else:
        DATA_DIR = '../datos_curados'
        OUT_TABLES = '../outputs/tablas'
        OUT_FIGS = '../outputs/figuras'
else:
    print("💻 Entorno Local detectado.")
    DATA_DIR = '../datos_curados' if os.path.exists('../datos_curados') else '05_Analisis_Reproducible/datos_curados'
    OUT_TABLES = '../outputs/tablas' if os.path.exists('../outputs/tablas') else '05_Analisis_Reproducible/outputs/tablas'
    OUT_FIGS = '../outputs/figuras' if os.path.exists('../outputs/figuras') else '05_Analisis_Reproducible/outputs/figuras'

os.makedirs(OUT_TABLES, exist_ok=True)
os.makedirs(OUT_FIGS, exist_ok=True)
print(f"Directorio de datos activo: {DATA_DIR}")

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
print("Librerías cargadas correctamente.")


In [ ]:
# Celda 2: Auto-Carga de Datos Curados
df_compuestos = pd.read_csv(os.path.join(DATA_DIR, 'compuestos_metadata.csv'))
df_conc = pd.read_csv(os.path.join(DATA_DIR, 'concentraciones_ng_g.csv'))
df_muestras = pd.read_csv(os.path.join(DATA_DIR, 'muestras_metadata.csv'))

home_idxs = df_muestras[df_muestras['microenvironment'] == 'House'].index
df_home_conc = df_conc.loc[home_idxs].copy()

print(f"Cargadas {len(df_home_conc)} muestras residenciales para la simulación de riesgo.")


In [ ]:
# Celda 3: Definición del Motor Vectorial Monte Carlo (N = 10,000 iteraciones)
np.random.seed(42)
N_ITER = 10000

# Toddlers (1-3 años)
bw_toddler = np.random.normal(loc=12.0, scale=1.5, size=N_ITER)
bw_toddler = np.clip(bw_toddler, 8.0, 18.0)

ir_toddler = np.random.lognormal(mean=np.log(50.0), sigma=0.3, size=N_ITER) / 1000.0
ir_toddler = np.clip(ir_toddler, 0.02, 0.20)

bsa_toddler = np.random.normal(loc=2800.0, scale=300.0, size=N_ITER)
af_toddler = np.random.uniform(low=0.04, high=0.20, size=N_ITER)

# Adultos
bw_adult = np.random.normal(loc=70.0, scale=8.0, size=N_ITER)
bw_adult = np.clip(bw_adult, 50.0, 100.0)

ir_adult = np.random.lognormal(mean=np.log(20.0), sigma=0.3, size=N_ITER) / 1000.0
ir_adult = np.clip(ir_adult, 0.005, 0.10)

bsa_adult = np.random.normal(loc=5700.0, scale=500.0, size=N_ITER)
af_adult = np.random.uniform(low=0.01, high=0.07, size=N_ITER)

print(f"Parámetros de Monte Carlo inicializados ({N_ITER:,} iteraciones).")


In [ ]:
# Celda 4: Simulación de EDI e Índices de Peligro (HQ) para TBOEP y TCIPP
toxicology_rfd = {
    'TBOEP': 15000.0,
    'TCIPP': 10000.0,
    'TCEP' : 22000.0,
    'TDCIPP': 15000.0,
    'DINCH': 1000000.0
}

c_tboep_samples = df_home_conc['TBOEP'].values
c_tboep_sim = np.random.choice(c_tboep_samples, size=N_ITER, replace=True)

edi_ing_toddler = (c_tboep_sim * ir_toddler) / bw_toddler
edi_ing_adult = (c_tboep_sim * ir_adult) / bw_adult

hq_tboep_toddler = edi_ing_toddler / toxicology_rfd['TBOEP']
hq_tboep_adult = edi_ing_adult / toxicology_rfd['TBOEP']

print(f"=== TBOEP RIESGO NO CANCERÍGENO (HQ) ===")
print(f"Toddlers: Mediana = {np.median(hq_tboep_toddler):.5f} | Percentil 95 = {np.percentile(hq_tboep_toddler, 95):.5f} (Seguro: HQ < 1)")
print(f"Adults  : Mediana = {np.median(hq_tboep_adult):.5f} | Percentil 95 = {np.percentile(hq_tboep_adult, 95):.5f} (Seguro: HQ < 1)")


In [ ]:
# Celda 5: Generación de la Figura 5 (Funciones de Densidad de Probabilidad de HQ)
fig, ax = plt.subplots(figsize=(8, 5))

sns.kdeplot(hq_tboep_toddler, label='Toddlers (1-3 yrs)', color='#E74C3C', fill=True, alpha=0.35, ax=ax)
sns.kdeplot(hq_tboep_adult, label='Adults', color='#2980B9', fill=True, alpha=0.35, ax=ax)

ax.axvline(1.0, color='darkred', linestyle='--', linewidth=2, label='Safety Threshold (HQ = 1.0)')
ax.set_xlabel('Hazard Quotient (HQ) for TBOEP', fontweight='bold')
ax.set_ylabel('Probability Density', fontweight='bold')
ax.set_title('Probabilistic Health Risk Assessment (Monte Carlo N = 10,000)', fontweight='bold')
ax.set_xlim(0, 0.5)
ax.legend()
ax.grid(True, linestyle=':', alpha=0.5)

fig5_path = os.path.join(OUT_FIGS, 'figura_5_probabilistic_risk_density.png')
plt.savefig(fig5_path)
plt.show()
print(f"Figura 5 guardada en: {fig5_path}")
